# Complete all five detectors across the three additional-subset preprocessing blocks.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
PROJECT_DIR = PROJECT_ROOT
REVISION_ROOT = PROJECT_DIR / "revision_outputs"
DATA_DIR = PROJECT_DIR / "data"
PYTHON_EXE = str(RELEASE_CLASSICAL_PYTHON)


In [ ]:
import subprocess,sys,os,time,json,platform
from pathlib import Path
from datetime import datetime,timezone


def python_candidates(explicit=None):
    if explicit:
        return [str(Path(explicit).expanduser().absolute())]
    candidates=[Path(sys.executable)]
    bases={Path(sys.prefix),Path(sys.base_prefix),Path(sys.executable).parent.parent,
           Path('/opt/anaconda3'),Path('/opt/miniconda3'),Path.home()/'miniconda3',Path.home()/'anaconda3'}
    for base in sorted(bases):
        candidates.extend(sorted((base/'envs').glob('*/bin/python')))
    candidates.extend(sorted((Path.home()/'.conda/envs').glob('*/bin/python')))
    registry=Path.home()/'.conda/environments.txt'
    if registry.is_file():
        candidates.extend(Path(line.strip())/'bin/python' for line in registry.read_text().splitlines() if line.strip())
    for base in [Path.cwd(),Path.cwd().parent]:
        candidates.extend([base/'.venv/bin/python',base/'venv/bin/python'])

    unique=[]
    for path in candidates:
        value=str(path.expanduser().absolute())
        if path.is_file() and value not in unique:unique.append(value)
    return unique


def compatible_python(expected,out,explicit=None):
    code="""import json,platform,importlib.metadata as m
v={'python':platform.python_version()}
for key,package in [('numpy','numpy'),('pandas','pandas'),('sklearn','scikit-learn'),('tensorflow','tensorflow'),('keras','keras'),('scipy','scipy'),('joblib','joblib')]:
 try:v[key]=m.version(package)
 except m.PackageNotFoundError:v[key]='NOT_INSTALLED'
print(json.dumps(v))
"""
    reports=[]
    for executable in python_candidates(explicit):
        item={'executable':executable}
        try:
            proc=subprocess.run([executable,'-c',code],capture_output=True,text=True,timeout=20)
            item['metadata_returncode']=proc.returncode
            if proc.returncode!=0:
                item['status']='METADATA_FAILED';item['stderr_tail']=proc.stderr[-1500:];reports.append(item);continue
            versions=json.loads(proc.stdout.strip().splitlines()[-1]);item['versions']=versions
            if any(versions.get(k)!=v for k,v in expected.items()):
                item['status']='VERSIONS_DIFFER';reports.append(item);continue
            print('Eğitim sürümleriyle eşleşen Python kontrol ediliyor:',executable,flush=True)
            probe="import numpy, pandas; import tensorflow as tf; import sklearn; print('TF_PROBE_OK',float(tf.reduce_sum(tf.constant([1.,2.])).numpy()),flush=True)"
            proc=subprocess.run([executable,'-X','faulthandler','-c',probe],capture_output=True,text=True,timeout=60)
            item.update(probe_returncode=proc.returncode,probe_stdout_tail=proc.stdout[-1500:],probe_stderr_tail=proc.stderr[-3000:])
            if proc.returncode==0 and 'TF_PROBE_OK 3.0' in proc.stdout:
                item['status']='READY';reports.append(item);return executable,reports
            item['status']='IMPORT_OR_TENSOR_TEST_FAILED'
        except subprocess.TimeoutExpired:
            item['status']='PROBE_TIMEOUT'
        except (OSError,ValueError) as e:
            item.update(status='PROBE_ERROR',error=str(e))
        reports.append(item)
    return None,reports




import hashlib,fcntl,copy,statistics,csv,math
from collections import defaultdict
METRICS=['AP','ROC_AUC','precision','recall','F1','Macro_F1']
METHODS=['DBSCAN','IF','LOF','OC-SVM','kNN']
PAIRS={(s,f) for s in [42,43,44] for f in range(1,11)}

def digest(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def canonical(x):return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False,separators=(',',':'),allow_nan=False).encode()).hexdigest()
def read(p):return json.loads(Path(p).read_text())
def require(ok,message):
    if not ok:raise ValueError(message)
def write(p,x):
    temp=p.with_suffix('.tmp');temp.write_text(json.dumps(x,ensure_ascii=False,indent=2,allow_nan=False));temp.replace(p)
def manifest(folder,hashes):
    require(bool(hashes),'Boş dosya doğrulama listesi')
    for name,h in hashes.items():
        require(Path(name).name==name,'Geçersiz çıktı adı')
        require((folder/name).is_file() and digest(folder/name)==h,'Çıktı eksik/değişmiş: '+str(folder/name))
def check_sources(root,data,job):
    for rel,h in job['known_hashes'].items():
        path=root/rel
        require(path.is_file() and digest(path)==h,'Kaynak dosya eksik/değişmiş: '+str(path))
    require(digest(data/f"train_{job['dataset']}.txt")==job['train_sha256'],'Ham veri değişmiş')
def validate(result,out,job,plan_hash):
    require(result['status']=='PASS_FOLD','Ek veri işi tamamlanmadı: '+result['status'])
    require(result['full_plan_sha256']==plan_hash,'Plan farklı')
    require(result['addendum_sha256']==PLAN['addendum_sha256'],'Detektör protokolü farklı')
    require(result['versions']==PLAN['expected_versions'],'Sürümler farklı')
    require((result['seed'],result['fold'],result['representation'])==(job['seed'],job['fold'],job['representation']),'İş kimliği farklı')
    require(result['baseline_reproduction_status']=='PASS_SINGLE_FOLD_REPRODUCTION','Önceki skorlar doğrulanmadı')
    require(result['PCA_refitted'] is False,'PCA yeniden fit edilmiş')
    require(not result['AE_training_performed'] and not result['test_files_read'] and not result['previous_outputs_modified'],'Beklenmeyen eğitim/veri/yazma')
    require(all(result[k]==job[k] for k in ['dataset','preprocessing']),'Blok farklı')
    rows=result['primary_results']+result['sensitivity_results']
    require(all(all(r[k]==job[k] for k in ['dataset','preprocessing']) for r in rows),'Satır bloğu farklı')
    keys=[(r['representation'],r['method'],r['RUL_window']) for r in rows]
    expected={(job['representation'],m,w) for m in METHODS+(['MSE'] if job['representation'].startswith('AE_') else []) for w in [20,30,40]}
    require(len(keys)==len(expected) and set(keys)==expected,'Skor hattı / RUL penceresi eksik')
    require(all((r['seed'],r['fold'])==(job['seed'],job['fold']) for r in rows),'Metrik fold/seed farklı')
    require(len(result['selection_trials'])==18 and all(t['status']=='VALID' for t in result['selection_trials']),'Eksik/geçersiz aday')
    for old in job['approved_primary_results']:
        actual=next(r for r in result['primary_results'] if r['method']==old['method'])
        require(all(actual[k]==v for k,v in old.items()),'Özgün10 ana sonucu değişmiş')
    manifest(out,result['output_sha256'])
def find_summary(root,glob,expected):
    for path in sorted(root.glob(glob)):
        try:value=read(path)
        except (OSError,ValueError):continue
        if canonical(value)==expected:return path,value
    raise FileNotFoundError('Onaylanan çıktı bulunamadı: '+glob+'; önceki çıktı klasörlerini koruyun.')
def validate_rows(rows):
    groups=defaultdict(list)
    for r in rows:
        for m in METRICS:require(math.isfinite(r[m]) and 0<=r[m]<=1,'Geçersiz metrik')
        tp,fp,tn,fn=[r[k] for k in ['TP','FP','TN','FN']]
        require(all(v>=0 and int(v)==v for v in [tp,fp,tn,fn]),'Geçersiz karmaşıklık matrisi')
        div=lambda n,d:n/d if d else 0
        vals={'precision':div(tp,tp+fp),'recall':div(tp,tp+fn),'F1':div(2*tp,2*tp+fp+fn),'Macro_F1':(div(2*tp,2*tp+fp+fn)+div(2*tn,2*tn+fp+fn))/2}
        require(all(abs(r[k]-v)<1e-10 for k,v in vals.items()),'Metrik/matris uyuşmazlığı')
        groups[r['dataset'],r['preprocessing'],r['representation'],r['method'],r['seed'],r['fold']].append(r)
    for group in groups.values():
        require(len(group)==3 and {r['RUL_window'] for r in group}=={20,30,40},'Duyarlılık kapsamı eksik')
        require(len({r['threshold'] for r in group})==1 and len({canonical(r['selected_params']) for r in group})==1,'Duyarlılıkta eşik/parametre değişmiş')
def aggregate(results, baseline):
    fields = ['dataset','preprocessing','representation','method']
    rows = [row for j in results for row in j['primary_results'] + j['sensitivity_results']]
    require(len(results)==450 and len(rows)==7290,'450 iş ve 7290 satır gerekli')
    validate_rows(rows)
    groups = defaultdict(list)
    for row in rows:
        key = tuple(row[k] for k in fields) + (row['RUL_window'],)
        groups[key].append(row)
    expected={(b['dataset'],b['preprocessing'],rep,m,w) for b in baseline['analysis_blocks'] for rep in baseline['representations'] for m in METHODS+(['MSE'] if rep.startswith('AE_') else []) for w in [20,30,40]}
    require(set(groups)==expected and len(expected)==243,'Veri bloğu/temsil/metot kapsamı eksik')
    overall=[]; per_seed=[]; index={}
    for key,group in sorted(groups.items()):
        require(len(group)==30 and {(r['seed'],r['fold']) for r in group}==PAIRS,'Fold/seed kapsamı eksik')
        ident=dict(zip(fields+['RUL_window'],key)); a=dict(**ident,runs=30)
        for m in METRICS:
            values=[r[m] for r in group]; a[m+'_mean']=statistics.mean(values); a[m+'_descriptive_sd']=statistics.stdev(values)
        overall.append(a)
        for seed in [42,43,44]:
            q=dict(**ident,seed=seed); subset=[r for r in group if r['seed']==seed]
            for m in METRICS:
                values=[r[m] for r in subset];q[m+'_fold_mean']=statistics.mean(values);q[m+'_fold_sd']=statistics.stdev(values)
            per_seed.append(q)
        if key[-1]==30:
            for r in group:index[key[:-1]+(r['seed'],r['fold'])]=r
    by_key={tuple(a[k] for k in fields+['RUL_window']):a for a in overall}
    for old in baseline['primary_results']+baseline['sensitivity_results']:
        new=by_key[tuple(old[k] for k in fields+['RUL_window'])]
        for m in METRICS:
            for suffix in ['_mean','_descriptive_sd']:require(abs(old[m+suffix]-new[m+suffix])<1e-10,'Özgün10 sonuçları değişmiş')
    def paired(left,right,metadata):
        result=dict(**metadata,pairs=30,direction='left minus right')
        for m in ['AP','F1','Macro_F1']:
            diff=[index[left+(s,f)][m]-index[right+(s,f)][m] for s,f in sorted(PAIRS)]
            result['delta_'+m+'_mean']=statistics.mean(diff);result['delta_'+m+'_descriptive_sd']=statistics.stdev(diff)
            result['positive_'+m+'_runs']=sum(v>1e-12 for v in diff);result['negative_'+m+'_runs']=sum(v< -1e-12 for v in diff);result['tied_'+m+'_runs']=sum(abs(v)<=1e-12 for v in diff)
        return result
    contrasts=[(rep,'Raw') for rep in baseline['representations'] if rep!='Raw']+[('AE_deep_d4','PCA_d4'),('AE_deep_d4','PCA_scaled_d4'),('AE_deep_d4','AE_deep_d21')]
    rep_pairs=[];mode_pairs=[]
    for b in baseline['analysis_blocks']:
        ds,mode=b['dataset'],b['preprocessing']
        for left,right in contrasts:
            for m in METHODS:rep_pairs.append(paired((ds,mode,left,m),(ds,mode,right,m),dict(**b,left=left,right=right,method=m)))
    for rep in baseline['representations']:
        for m in METHODS+(['MSE'] if rep.startswith('AE_') else []):
            mode_pairs.append(paired(('FD002','conditionwise',rep,m),('FD002','global',rep,m),dict(dataset='FD002',representation=rep,method=m,left='conditionwise',right='global')))
    return dict(primary_results=[a for a in overall if a['RUL_window']==30],sensitivity_results=[a for a in overall if a['RUL_window']!=30],per_seed_results=per_seed,paired_primary_contrasts=rep_pairs,paired_FD002_preprocessing=mode_pairs,fold_primary_results=[r for r in rows if r['RUL_window']==30],fold_sensitivity_results=[r for r in rows if r['RUL_window']!=30],primary_groups=81,metric_rows=len(rows),source10_aggregates_preserved=True)

def write_csv(path,rows):
    keys=list(dict.fromkeys(k for r in rows for k in r))
    with path.open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=keys);w.writeheader()
        for r in rows:w.writerow({k:json.dumps(v,ensure_ascii=False) if isinstance(v,(dict,list)) else v for k,v in r.items()})
def run_child(exe,worker,cp,attempt,label):
    with (attempt/'worker.log').open('w') as log:
        proc=subprocess.Popen([exe,'-u','-X','faulthandler',str(worker),str(cp)],stdout=log,stderr=subprocess.STDOUT)
        try:
            last=time.monotonic()
            while proc.poll() is None:
                time.sleep(1)
                if time.monotonic()-last>30:print(label,'çalışıyor; ayrıntı:',attempt/'worker.log',flush=True);last=time.monotonic()
        except BaseException:
            proc.terminate()
            try:proc.wait(timeout=10)
            except subprocess.TimeoutExpired:proc.kill();proc.wait()
            raise
    sp=attempt/'fold_completion_summary.json'
    require(proc.returncode==0 and sp.is_file(),'Alt işlem durdu; worker.log: '+str(attempt))
    return read(sp),sp

def run_full():
    root=REVISION_ROOT.expanduser().resolve();data=DATA_DIR.expanduser().resolve()
    ph=canonical(PLAN);out=root/'10C_FD002_FD003_five_method_full'/ph[:20];out.mkdir(parents=True,exist_ok=True)
    sp=out/'additional_completion_summary.json'
    summary=dict(stage=PLAN['stage'],status='RUNNING',full_plan_sha256=ph,addendum_sha256=PLAN['addendum_sha256'],expected_jobs=450,completed_jobs=0,pilot_jobs_reused=0,new_jobs_started_this_invocation=0,PCA_refitted=False,AE_training_performed=False,test_files_read=False,previous_outputs_modified=False,job_sources=[])
    start=time.monotonic()
    with (out/'run.lock').open('a') as lock:
        fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
        try:
            require(hashlib.sha256(WORKER_SOURCE.encode()).hexdigest()==PLAN['worker_sha256'],'Worker değişmiş')
            write(out/'full_run_plan.json',PLAN);worker=out/'worker.py';worker.write_text(WORKER_SOURCE)
            pp,pilot=find_summary(root,'10B1_FD002_FD003_five_method_pilot/*/additional_pilot_summary.json',PLAN['pilot_summary_canonical_sha256'])
            require(pilot['status']=='PASS_ADDITIONAL_FIVE_METHOD_PILOT' and canonical(read(pp.parent/'pilot_plan.json'))==PLAN['pilot_plan_sha256'],'Pilot kimliği farklı')
            relative=Path(PLAN['jobs'][0]['fold_relative_directory'])
            source=root/relative.parts[0]/relative.parts[1];baseline=read(source/'additional_cv_summary.json')
            require(canonical(baseline)==PLAN['baseline10_sha256'],'Özgün10 özeti değişmiş');manifest(source,baseline['output_file_sha256'])
            summary.update(pilot_summary=str(pp),versions=PLAN['expected_versions'],resource_amendment=PLAN['resource_amendment'])
            exe,reports=compatible_python(PLAN['expected_versions'],out,PYTHON_EXE);summary.update(worker_executable=exe,environment_checks=reports)
            require(exe is not None,'Uyumlu Python bulunamadı')
            results=[]
            for n,job in enumerate(PLAN['jobs'],1):
                check_sources(root,data,job)
                label=f"{n}/450 {job['job_key']} seed{job['seed']} fold{job['fold']}"
                is_pilot=job['seed']==42 and job['fold']==1
                if is_pilot:
                    jd=pp.parent/job['job_key'];pointer=read(jd/'done.json');attempt=jd/pointer['attempt']
                    require(attempt.parent==jd,'Pilot kayıt yolu farklı')
                    jsp=attempt/'fold_completion_summary.json';require(digest(jsp)==pointer['summary_sha256'],'Pilot iş özeti değişmiş')
                    result=read(jsp);validate(result,attempt,job,PLAN['pilot_plan_sha256'])
                    approved=next(r for r in pilot['results'] if all(r[k]==job[k] for k in ['dataset','preprocessing','representation']))
                    require(result==approved,'Pilot sonucu paylaşılan özetten farklı')
                    kind='approved10B1_reused';summary['pilot_jobs_reused']+=1;print('PİLOTTAN:',label,flush=True)
                else:
                    jd=out/job['job_key']/f"seed_{job['seed']}"/f"fold_{job['fold']:02d}";jd.mkdir(parents=True,exist_ok=True);done=jd/'done.json'
                    if done.is_file():
                        pointer=read(done);attempt=jd/pointer['attempt'];require(attempt.parent==jd,'Geçersiz kayıt yolu')
                        jsp=attempt/'fold_completion_summary.json';require(digest(jsp)==pointer['summary_sha256'],'Tamamlanmış özet değişmiş')
                        result=read(jsp);validate(result,attempt,job,ph);kind='completed10C_reused';print('KAYITTAN:',label,flush=True)
                    else:
                        attempt=jd/('attempt_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'));attempt.mkdir()
                        config={**copy.deepcopy(job),'revision_root':str(root),'data_directory':str(data),'output_directory':str(attempt),'approved09_summary':PLAN['approved09_summary'],'expected_versions':PLAN['expected_versions'],'five_method_addendum':PLAN['five_method_addendum'],'addendum_sha256':PLAN['addendum_sha256'],'full_plan_sha256':ph}
                        cp=attempt/'config.json';write(cp,config);summary['new_jobs_started_this_invocation']+=1;print('ÇALIŞIYOR:',label,flush=True)
                        result,jsp=run_child(exe,worker,cp,attempt,label);summary['last_job_status']=result['status'];summary['last_job_summary']=str(jsp)
                        if result.get('status')!='PASS_FOLD':
                            summary['failed_job']={'job':label,'status':result.get('status'),'error':result.get('error'),'traceback':result.get('traceback'),'log_directory':str(attempt)}
                        validate(result,attempt,job,ph);write(done,{'attempt':attempt.name,'summary_sha256':digest(jsp)});kind='new10C_completed'
                results.append(result);summary['completed_jobs']=len(results)
                summary['job_sources'].append(dict(dataset=job['dataset'],preprocessing=job['preprocessing'],representation=job['representation'],seed=job['seed'],fold=job['fold'],kind=kind,summary_path=str(jsp),summary_sha256=digest(jsp)))
                write(sp,summary)
            summary.update(aggregate(results,baseline))
            trials=[dict(t,dataset=r['dataset'],preprocessing=r['preprocessing'],seed=r['seed'],fold=r['fold']) for r in results for t in r['selection_trials']]
            require(len(trials)==8100 and all(t['status']=='VALID' for t in trials),'8100 geçerli aday gerekli')
            summary.update(status='PASS_ADDITIONAL_CV_5_METHODS',full_additional_comparison_completed=True,pilot_jobs_reused=15,new_AE_training_runs=0,total_candidate_trials_including_pilot=8100,selection_trials=trials,selected_configurations=[dict(dataset=r['dataset'],preprocessing=r['preprocessing'],seed=r['seed'],fold=r['fold'],representation=r['representation'],configurations=r['selected_configurations']) for r in results],statistical_tests_performed=False,aggregation='Equal-weight fold/seed means and descriptive sample SD; no independent-30-samples inference, no pooled-score AP.',limits=['FD002/FD003 late-life proxy comparison; final-detector SHAP is a separate analysis.','RUL-derived late-life proxy is not annotated anomaly onset.','Follow-up representation comparison; no outer-fold winner selection claimed.','DBSCAN core-distance extension and unequal candidate search budgets explicitly retained.'])
            write_csv(out/'all_fold_metrics.csv',summary['fold_primary_results']+summary['fold_sensitivity_results'])
            write_csv(out/'aggregate_metrics.csv',summary['primary_results']+summary['sensitivity_results'])
            write_csv(out/'per_seed_metrics.csv',summary['per_seed_results']);write_csv(out/'paired_contrasts.csv',summary['paired_primary_contrasts'])
            write_csv(out/'paired_FD002_preprocessing.csv',summary['paired_FD002_preprocessing'])
            summary['output_sha256']={name:digest(out/name) for name in ['full_run_plan.json','worker.py','all_fold_metrics.csv','aggregate_metrics.csv','per_seed_metrics.csv','paired_contrasts.csv','paired_FD002_preprocessing.csv']}
        except KeyboardInterrupt:summary.update(status='INTERRUPTED',error='Run All ile doğrulanmış başarılı işler yeniden kullanılabilir.')
        except Exception as e:
            import traceback
            summary.update(status='REVIEW_REQUIRED',error=str(e),traceback=traceback.format_exc());traceback.print_exc()
        finally:
            summary['elapsed_seconds_this_invocation']=time.monotonic()-start;write(sp,summary)
    print('\nDURUM:',summary['status']);print('GÖNDERİLECEK DOSYA:',sp)
    return summary,sp


In [ ]:
PLAN = load_json('assets/10C/03_plan.json')
WORKER_SOURCE = load_text('assets/10C/03_worker_source.py')


In [ ]:
result, summary_path = run_full()
